
# Going from wide to long


In data analysis, unpivoting (often referred to as "melting") is the process of transforming a dataset from a wide format—where data is spread across multiple columns—into a long (or narrow) format, where those column headers become values in a single row. This is a crucial step for "tidying" data, as most modern visualization tools and statistical models prefer one observation per row and one variable per column.

In Pandas, this is primarily handled by the `melt()` method. When you unpivot, you distinguish between identifier variables (columns you want to keep as-is, like 'Name' or 'Date') and measured variables (the columns you want to collapse). The function then stacks the measured columns into two new columns: one containing the original column names (the "variable") and another containing the values themselves.

Another powerful option is to use the `.stack()` method, which approaches the problem from a structural perspective rather than a column-naming one. While `melt()` "liquefies" your data by targeting specific columns, `stack()` works by "compressing" the existing column headers down into the row index. It effectively pivots the innermost level of the column labels to become the innermost level of the row index, resulting in a MultiIndex Series, which you can then reset into a data frame.



From https://stackoverflow.com/a/48136002


## Setup



In [ ]:
from io import StringIO
import pandas as pd
import markdown
import duckdb


### Convert between data formats

#### Markdown/Text


In [ ]:
# Assign Markdown string to a variable
table_md = '''
|Date|A|B|C|D|E |
|----|-|-|-|-|--|
|2005|1|2|3|4|50|
|2006|6|7|8|9|10|
'''
table_md


In [ ]:
print(table_md)

#### HTML


In [ ]:
# Convert Markdown to HTML
table_html=markdown.markdown(table_md, extensions=['markdown.extensions.tables'])
table_html


In [ ]:
%%html
<table>
<thead>
<tr>
<th>Date</th>
<th>A</th>
<th>B</th>
<th>C</th>
<th>D</th>
<th>E</th>
</tr>
</thead>
<tbody>
<tr>
<td>2005</td>
<td>1</td>
<td>2</td>
<td>3</td>
<td>4</td>
<td>50</td>
</tr>
<tr>
<td>2006</td>
<td>6</td>
<td>7</td>
<td>8</td>
<td>9</td>
<td>10</td>
</tr>
</tbody>
</table>

#### Data Frame

In [ ]:
# Convert HTML to a DataFrame

df = pd.read_html(StringIO(table_html))[0]
df


## Using .stack()

#### "Protect" columns by converting to index

In [ ]:
(
  df
    .set_index('Date')
)

#### Stack - converting columns to key-value pairs

In [ ]:
(
  df
    .set_index('Date')
    .stack()
)

In [ ]:
(
  df
    .set_index('Date')
    .stack()
    .rename('Value')
)

In [ ]:
(
  df
    .set_index('Date')
    .stack()
    .rename('Value')
    .rename_axis(['Date','Category'])
)

In [ ]:
df_stack = (
  df
  .set_index('Date')
  .stack()
  .rename('Value')
  .rename_axis(['Date','Category'])
  .reset_index()
)
df_stack


## Using .melt()

### List of columns to "keep"

In [ ]:
keeps = df.columns[:1]
keeps

### Melt


In [ ]:
df_melt = df.melt(
  keeps,
  var_name='Category',
  value_name='Val'
)
df_melt


### Sorted by "Date", then "Category"

In [ ]:
df_melt2 = (
  df
    .melt(
      keeps,
      var_name='Category',
      value_name='Val'
    )
    .sort_values(['Date','Category'])
    .reset_index(drop=True)
)
df_melt2


# Going from long to wide


Which process you use depends on the initial format and the desired final format.


## Using `.pivot()`

Pivoting takes at least three columns, converting the first into an index, the categories into the second as the columns, and the values in the third as the corresponding values for the columns.

We can use pivot to reverse the process.  Notice that we have to turn the "Date" column into an index.  

In [ ]:
(
df_melt2
  .pivot(
    index=['Date'],
    columns=['Category'],
    values='Val',
  )
)


We can fix that by reseting and renaming the indices.

In [ ]:
(
df_melt2
  .pivot(
    index=['Date'],
    columns=['Category'],
    values='Val')
  .reset_index()
  .rename_axis(None, axis=1)
)


In [ ]:
ct = df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").drop(columns=["id"])
ct

In [ ]:
ct["fruit"].unique()

In [ ]:
ct.value_counts()

In [ ]:
df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").value_counts()

In [ ]:
df.assign(fruit=df.fruit.str.split(", ")).explode("fruit").value_counts().reset_index()

# Census


In [ ]:
url = "https://www2.census.gov/programs-surveys/acs/summary_file/2024/table-based-SF/data/5YRData/acsdt5y2024-b01001.dat"
url


In [ ]:
!curl -s {url} | head -2

In [ ]:
df = pd.read_csv(url, sep='|', nrows=None)
df.shape


In [ ]:
df.head(10)

In [ ]:
df_long = df.melt(
    id_vars = df.columns[:1],
    value_vars= df.columns[1:],
)
df_long.shape


In [ ]:
f"{df_long.shape[0]:,}"

In [ ]:
df_long.head()

In [ ]:
# for next time:
# ^([^_]+)_([A-Z])(\d+)$

# DuckDB can see the 'df_long' variable in your local scope automatically
sql = """
SELECT
    *,
    regexp_extract(variable, '^(.+)_(.)(.+)$', 1) AS id_part,
    regexp_extract(variable, '^(.+)_(.)(.+)$', 2) AS type_letter,
    regexp_extract(variable, '^(.+)_(.)(.+)$', 3) AS number_part
FROM df_long
"""

# Execute and turn back into a parquet file
duckdb.query(sql).to_parquet("parsed_data.parquet")


In [ ]:
!ls -la --si


In [ ]:
duckdb.query("select * from parsed_data.parquet limit 10").df()

In [ ]:
sql = """
with sample as (
  select *
  FROM parsed_data.parquet
)

PIVOT (
    SELECT
        GEO_ID,
        id_part,
        number_part,
        type_letter,
        value
    FROM sample
)
ON type_letter
USING CAST(SUM(value) AS INTEGER)
GROUP BY GEO_ID, id_part, number_part
ORDER BY GEO_ID, id_part, number_part
"""

df_wide = duckdb.query(sql).df()
df_wide


In [ ]:
df_wide.info(memory_usage='deep')


In [ ]:
duckdb.query("""
    COPY (
        SELECT *,
               LEFT(split_part(GEO_ID, 'US', 2), 2) as state_fips
        FROM df_wide
    )
    TO 'census_data_partitioned'
    (FORMAT PARQUET, PARTITION_BY (state_fips, id_part), OVERWRITE_OR_IGNORE 1)
""")


In [ ]:
!ls -la

In [ ]:
!du -ms census_data_partitioned/

In [ ]:
duckdb.query('''
select
  GEO_ID,
  state_fips,
  id_part,
  number_part,
  E as estimate,
  M as margin_of_error
from read_parquet('census_data_partitioned/**/*.parquet', hive_partitioning=True)
limit 10
''')